In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler


DATA_PATH = Path("data") / "data.xlsx"
OUTPUT_DIR = Path("outputs") / "figure_s7"
WORKBOOK_FILENAME = "brand_snv_results.xlsx"
N_PCS = 5

GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16",
    "SH19", "SH20", "T3", "T6", "T9", "T12", "T15", "Te3", "Te6",
    "Te9", "Te12", "Te15",
}

CASE_ROOTS = {
    "95_gasoline": GASOLINE_95_ROOTS,
    "98_gasoline": GASOLINE_98_ROOTS,
    "diesel": DIESEL_ROOTS,
}

CASE_NAMES = {
    "95_gasoline": "#95 Gasoline",
    "98_gasoline": "#98 Gasoline",
    "diesel": "Diesel",
}

IMAGE_FILENAMES = {
    "95_gasoline": "#95_Gasoline_external_validation_ld1_ld2_snv.png",
    "98_gasoline": "#98_Gasoline_external_validation_ld1_ld2_snv.png",
    "diesel": "Diesel_external_validation_ld1_ld2_snv.png",
}

PANEL_LABELS = {
    "95_gasoline": "a)",
    "98_gasoline": "b)",
    "diesel": "c)",
}

BRAND_ORDER = ["TinQ", "Texaco", "Shell"]

EXTERNAL_TEST_ROOTS = {
    "T1", "T2", "T3", "T4", "T5", "T6",
    "Te1", "Te2", "Te3", "Te4", "Te5", "Te6",
    "SH1", "SH2", "SH3", "SH4", "SH5", "SH6", "SH7", "SH8",
}


def root_from_id(sample_id: str) -> str:
    return str(sample_id).split("-", 1)[0]


def brand_from_root(root: str) -> str | None:
    if root.startswith("Te"):
        return "Texaco"
    if root.startswith("SH"):
        return "Shell"
    if root.startswith("T"):
        return "TinQ"
    return None


def sort_spectral_columns(df: pd.DataFrame) -> pd.DataFrame:
    numeric_columns = []
    wavelengths = []

    for column in df.columns:
        try:
            wavelength = float(column)
        except (TypeError, ValueError):
            continue
        numeric_columns.append(column)
        wavelengths.append(wavelength)

    if not numeric_columns:
        raise RuntimeError("No numeric spectral columns were found in the Positive sheet.")

    order = np.argsort(np.asarray(wavelengths, dtype=float))
    sorted_columns = [numeric_columns[index] for index in order]
    return df.loc[:, sorted_columns].astype(float)


def load_positive_data() -> tuple[np.ndarray, pd.DataFrame]:
    spectra = pd.read_excel(DATA_PATH, sheet_name="Positive", index_col=0)
    spectra = spectra.sort_index()
    spectra.index = spectra.index.astype(str)
    spectra = sort_spectral_columns(spectra)

    metadata = pd.DataFrame(index=spectra.index)
    metadata["root"] = metadata.index.to_series().map(root_from_id)
    metadata["brand"] = metadata["root"].map(brand_from_root)
    return spectra.to_numpy(dtype=float), metadata


def split_training_and_external(
    all_spectra: np.ndarray,
    metadata: pd.DataFrame,
    case_key: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    roots = metadata["root"].to_numpy(dtype=object)
    brands = metadata["brand"].to_numpy(dtype=object)
    case_mask = np.isin(roots, list(CASE_ROOTS[case_key]))
    external_mask = case_mask & np.isin(roots, list(EXTERNAL_TEST_ROOTS))
    training_mask = case_mask & ~np.isin(roots, list(EXTERNAL_TEST_ROOTS))

    if not training_mask.any():
        raise RuntimeError(f"No training samples found for case: {case_key}")
    if not external_mask.any():
        raise RuntimeError(f"No external validation samples found for case: {case_key}")

    training_brands = brands[training_mask].astype(object)
    external_brands = brands[external_mask].astype(object)
    missing_training = [brand for brand in BRAND_ORDER if brand not in training_brands]
    missing_external = [brand for brand in BRAND_ORDER if brand not in external_brands]
    if missing_training:
        raise RuntimeError(f"Training set for {case_key} is missing brands: {missing_training}")
    if missing_external:
        raise RuntimeError(f"External validation set for {case_key} is missing brands: {missing_external}")

    return (
        all_spectra[training_mask],
        training_brands,
        all_spectra[external_mask],
        external_brands,
    )


def preprocess_snv(spectra: np.ndarray) -> np.ndarray:
    spectra = np.asarray(spectra, dtype=float)
    means = spectra.mean(axis=1, keepdims=True)
    standard_deviations = spectra.std(axis=1, ddof=1, keepdims=True)
    standard_deviations[standard_deviations == 0.0] = 1.0
    return (spectra - means) / standard_deviations


def fit_model(
    training_spectra: np.ndarray,
    training_brands: np.ndarray,
) -> tuple[StandardScaler, PCA, LinearDiscriminantAnalysis]:
    training_snv = preprocess_snv(training_spectra)
    scaler = StandardScaler(with_mean=True, with_std=True)
    training_scaled = scaler.fit_transform(training_snv)
    pca = PCA(n_components=N_PCS)
    training_scores = pca.fit_transform(training_scaled)
    lda = LinearDiscriminantAnalysis()
    lda.fit(training_scores, training_brands)
    return scaler, pca, lda


def make_explained_variance_table(pca: PCA) -> pd.DataFrame:
    explained = pca.explained_variance_ratio_
    return pd.DataFrame(
        {
            "PC": [f"PC{index}" for index in range(1, len(explained) + 1)],
            "Explained variance ratio (%)": explained * 100.0,
            "Cumulative explained variance (%)": np.cumsum(explained) * 100.0,
        }
    )


def project_external(
    external_spectra: np.ndarray,
    external_brands: np.ndarray,
    scaler: StandardScaler,
    pca: PCA,
    lda: LinearDiscriminantAnalysis,
) -> tuple[np.ndarray, pd.DataFrame, np.ndarray]:
    external_snv = preprocess_snv(external_spectra)
    external_scaled = scaler.transform(external_snv)
    external_pca_scores = pca.transform(external_scaled)
    predicted_brands = lda.predict(external_pca_scores)
    ld_scores = lda.transform(external_pca_scores)
    classification_results = pd.DataFrame(
        {"is_correct": external_brands == predicted_brands}
    )

    if hasattr(lda, "explained_variance_ratio_"):
        ld_percentages = np.asarray(lda.explained_variance_ratio_, dtype=float) * 100.0
    else:
        ld_variances = np.var(ld_scores, axis=0, ddof=1)
        total_variance = np.sum(ld_variances)
        if total_variance > 0:
            ld_percentages = ld_variances / total_variance * 100.0
        else:
            ld_percentages = np.full(ld_scores.shape[1], np.nan)

    return ld_scores, classification_results, ld_percentages


def summarise_classification(results: pd.DataFrame, case_name: str) -> pd.DataFrame:
    number_of_samples = len(results)
    number_correct = int(results["is_correct"].sum())

    return pd.DataFrame(
        [
            {
                "Case": case_name,
                "Preprocessing method": "SNV",
                "Dataset": "External validation",
                "Number of samples": number_of_samples,
                "Number of correctly classified samples": number_correct,
                "Classification accuracy": number_correct / number_of_samples,
            }
        ]
    )


def plot_external_ld1_ld2(
    ld_scores: np.ndarray,
    external_brands: np.ndarray,
    ld_percentages: np.ndarray,
    case_key: str,
) -> None:
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    ax.set_facecolor("#f5f5f5")

    for brand in BRAND_ORDER:
        mask = external_brands == brand
        if mask.any():
            ax.scatter(
                ld_scores[mask, 0],
                ld_scores[mask, 1],
                s=28,
                alpha=0.80,
                label=brand,
                edgecolors="none",
            )

    ld1_percentage = float(ld_percentages[0])
    ld2_percentage = float(ld_percentages[1])
    case_name = CASE_NAMES[case_key]
    ax.set_xlabel(
        f"LD1 ({ld1_percentage:.1f}% discriminative ability)",
        family="serif",
    )
    ax.set_ylabel(
        f"LD2 ({ld2_percentage:.1f}% discriminative ability)",
        family="serif",
        rotation=270,
        labelpad=22,
    )
    ax.set_title(
        f"{PANEL_LABELS[case_key]} LDA discrimination plot (LD1 vs LD2) "
        f"of the external validation – {case_name}",
        family="serif",
        pad=12,
    )
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()
    ax.yaxis.set_label_position("right")
    ax.yaxis.tick_right()
    ax.tick_params(
        axis="x",
        which="both",
        top=True,
        labeltop=True,
        bottom=False,
        labelbottom=False,
    )
    ax.tick_params(
        axis="y",
        which="both",
        right=True,
        labelright=True,
        left=False,
        labelleft=False,
    )
    ax.grid(True, color="#c8c8c8", linewidth=0.6, alpha=0.8)
    ax.spines["top"].set_linewidth(1.1)
    ax.spines["top"].set_color("black")
    ax.spines["right"].set_linewidth(1.1)
    ax.spines["right"].set_color("black")
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="both", which="both", direction="in", length=5)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / IMAGE_FILENAMES[case_key],
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def run() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_spectra, metadata = load_positive_data()
    summary_tables = []
    workbook_path = OUTPUT_DIR / WORKBOOK_FILENAME

    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        for case_key in ["95_gasoline", "98_gasoline", "diesel"]:
            training_spectra, training_brands, external_spectra, external_brands = (
                split_training_and_external(all_spectra, metadata, case_key)
            )
            scaler, pca, lda = fit_model(training_spectra, training_brands)
            explained_variance = make_explained_variance_table(pca)
            ld_scores, classification_results, ld_percentages = project_external(
                external_spectra,
                external_brands,
                scaler,
                pca,
                lda,
            )
            summary_tables.append(
                summarise_classification(classification_results, CASE_NAMES[case_key])
            )
            explained_variance.to_excel(
                writer,
                sheet_name=f"{case_key}_explained_var",
                index=False,
            )
            plot_external_ld1_ld2(
                ld_scores,
                external_brands,
                ld_percentages,
                case_key,
            )

        pd.concat(summary_tables, ignore_index=True).to_excel(
            writer,
            sheet_name="all_cases_summary",
            index=False,
        )

    print(f"Saved: {workbook_path}")


if __name__ == "__main__":
    run()
